# A plugin of your own

A platform is a `Plugin`: it declares what it can do and which secrets it
needs, and implements `post`, `read`, `reply`. Real ones are packages in
`packages/` registered through the entry point group `amoorion.plugins`
(`uv run task plugin:new -- <platform> <Class> "<description>"` starts
one). Here one is registered by hand, for this notebook only.

In [1]:
import os

from amoorion import Amoorion

# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
Amoorion.init()

PosixPath('/tmp/amoorion-x6mlrxnv')

In [2]:
from datetime import UTC, datetime

from typing import ClassVar

from amoorion.models import Message, Post

from amoorion.plugins import PLUGINS, Capability, Plugin


# A whole platform: a name, what it can do, its secrets, and those operations.
class Notebook(Plugin):
    name: ClassVar[str] = "notebook"
    description: ClassVar[str] = "prints posts in the notebook"
    capabilities: ClassVar[frozenset[Capability]] = frozenset({Capability.POST})
    secrets: ClassVar[tuple[str, ...]] = ("greeting",)

    def post(
        self,
        post: Post,
    ) -> Message:

        text = f"{self.secret('greeting')}: {post.text}"

        print(text)

        return Message(
            platform=self.name,
            text=text,
            id="1",
            created_at=datetime.now(UTC),
        )


# In a package, an entry point does this by itself.
PLUGINS.register(Notebook)

__main__.Notebook

Its secret `notebook/greeting` comes from the vault like any other:

In [3]:
from amoorion.methods import Publish

# The secret it asked for.
os.environ["AMOORION_SECRET_NOTEBOOK_GREETING"] = "hello"

# It works like any other platform.
Publish()(
    text="from my own plugin",
    to=[
        "notebook",
        "dryrun",
    ],
)

hello: from my own plugin


[{'platform': 'notebook',
  'ok': True,
  'message': {'platform': 'notebook',
   'text': 'hello: from my own plugin',
   'id': '1',
   'url': None,
   'author': None,
   'reply_to': None,
   'created_at': '2026-10-31T07:55:18.221450Z'},
  'error': None},
 {'platform': 'dryrun',
  'ok': True,
  'message': {'platform': 'dryrun',
   'text': 'from my own plugin',
   'id': '1',
   'url': None,
   'author': None,
   'reply_to': None,
   'created_at': '2026-10-31T07:55:18.227111Z'},
  'error': None}]

What it can't do it refuses with a clear error:

In [4]:
from amoorion.methods import Read

# It never said it could read, so it refuses.
try:
    Read()(platform="notebook")
except Exception as error:
    print(
        type(error).__name__,
        "-",
        error,
    )

Unsupported - notebook can't read


Test a real plugin against a fake of its platform's API (a local HTTP
server), as every package in `packages/` does; `uv run task test:live --
<platform>` tries it against the real account. Guide:
[docs/plugins.md](../docs/plugins.md).